In [1]:
print("Hola PySpark")

Hola PySpark


# INTRODUCCION BASICA A PYSPARK

In [1]:
# Celda de comprobacion: Java, JAVA_HOME, PySpark y los CSV.
# SPARK_LOCAL_IP / SPARK_LOCAL_HOSTNAME evitan que Spark tome el hostname
# que Docker Desktop mete en Windows: kubernetes.docker.internal
# (NO estamos en un cluster K8s: el master sera local[*]).

import os
import platform
import shutil
import socket
import sys
from pathlib import Path

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

RAIZ = Path.cwd().parent if Path.cwd().name in {"notebook", "notebooks"} else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Python   :", sys.version.split()[0], sys.executable)
print("Raiz     :", RAIZ)
print("Hostname :", socket.gethostname())
print("java     :", shutil.which("java") or "NO ESTA EN EL PATH")
print("JAVA_HOME:", os.environ.get("JAVA_HOME") or "(vacio: Spark suele fallar)")

try:
    import pandas as pd
    import pyarrow
    import pyspark

    print("PySpark  :", pyspark.__version__, "(pyspark.txt pide 3.5.6)")
    print("PyArrow  :", pyarrow.__version__, "(pyspark.txt pide 19.0.1)")
    print("Pandas   :", pd.__version__)
except ImportError as error:
    raise ImportError(
        "Falta PySpark o pyarrow en este kernel.\n"
        "  conda activate py_env\n"
        "  pip install -r requirements-pyspark.txt\n"
        "Y en Cursor: elige el kernel de py_env.\n"
        f"Detalle: {error}"
    ) from None

print("SO       :", platform.system(), platform.release())
print("Entorno listo. El hostname de Docker no implica Kubernetes.")

Python   : 3.11.15 c:\Users\JORGE\miniconda3\envs\py_env\python.exe
Raiz     : c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_2
Hostname : ECHAVARRIA
java     : C:\Program Files\Eclipse Adoptium\jdk-21.0.5.11-hotspot\bin\java.EXE
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.5.11-hotspot\
PySpark  : 4.2.0 (pyspark.txt pide 3.5.6)
PyArrow  : 19.0.1 (pyspark.txt pide 19.0.1)
Pandas   : 3.0.5
SO       : Windows 10
Entorno listo. El hostname de Docker no implica Kubernetes.


## Arquitectura básica y procesamiento dsitribuido

Spark no pone los 200.000 registros en un solo proceso de Python. Parte el trabajo: un **Driver** coordina y varios **Executors** calculan en paralelo.

| Pieza | Dónde vive | Qué hace | Qué NO hace |
|---|---|---|---|
| **Driver** | El proceso que arranca el notebook (`SparkSession`) | Crea el plan, reparte tareas, junta el resultado final | No recorre fila a fila los datos grandes |
| **Cluster Manager** | Local (esta clase), YARN o Kubernetes | Da CPU y RAM a los executors | No escribe el SQL ni el DataFrame |
| **Executor** | Proceso JVM worker (aquí, en tu misma máquina) | Lee su partición, filtra, agrega, escribe | No ve el dataset completo |
| **Tarea** | Dentro de un executor | Una unidad de trabajo sobre **una** partición | Si falla, Spark la reintenta en otro executor |

En local parece un solo programa. El modelo es el mismo que en un clúster: el Driver manda el plan; los Executors ejecutan; el Driver solo recibe el resumen (`count`, `show`, `collect`).

`collect()` trae **todo** al Driver. En clase con 200.000 filas aguanta; en millones, satura la memoria del Driver. Para mirar datos usa `show()` o `limit`.

## Transformaciones vs acciones

Es el concepto que más se examina. Spark es **perezoso**: `filter` y `groupBy` **no leen el CSV**. Solo dibujan un plan. El trabajo arranca cuando pides un **resultado**.

| | Transformación | Acción |
|---|---|---|
| ¿Qué devuelve? | Otro DataFrame (un plan nuevo) | Un valor a Python o datos a disco |
| ¿Cuándo corre? | **Nunca sola.** Se encadena (lazy) | **Ahora.** Dispara el job |
| ¿Dónde calcula? | Executors, cuando llegue la acción | Executors calculan; el Driver recibe el final |
| Ejemplos | `select`, `filter`, `withColumn`, `join`, `groupBy`, `agg`, `dropDuplicates` | `show`, `count`, `collect`, `take`, `head`, `write.csv`, `toPandas` |
| Truco de examen | Encadenar 10 transformaciones **no** abre el archivo 10 veces | Cada acción puede **recalcular** todo el plan (salvo `cache`) |

Cadena típica:

```
CSV  →  filter  →  groupBy  →  agg     →  show / count
         (plan)     (plan)     (plan)      (aquí sí corre)
```

Si llamas `count()` y luego `show()`, Spark puede leer los datos **dos veces**. `cache()` o `persist()` guarda el resultado intermedio en los executors para no repetir el trabajo.

In [2]:
# Arrancar Spark: una sesion LOCAL para toda la clase.
# El worker de Python DEBE ser el mismo interprete del kernel.
# Si no se configura, Windows intenta reconectar y aparece
# "Python worker failed to connect back".

import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

activa = SparkSession.getActiveSession()
if activa is not None:
    activa.stop()

try:
    spark = (
        SparkSession.builder
        .appName("Unidad2-Clase1-PySpark")
        .master("local[*]")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.execution.arrow.pyspark.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )
except Exception as error:
    raise RuntimeError(
        "No se pudo arrancar Spark.\n"
        "Comprueba Java, JAVA_HOME y que el kernel sea py_env.\n"
        f"Detalle: {type(error).__name__}: {error}"
    ) from None

spark.sparkContext.setLogLevel("WARN")

conf = spark.sparkContext.getConf()
print("SparkSession lista (modo LOCAL)")
print("  App      :", spark.sparkContext.appName)
print("  Master   :", spark.sparkContext.master)
print("  Version  :", spark.version)
print("  Driver   :", conf.get("spark.driver.host"))
print("  UI       :", spark.sparkContext.uiWebUrl)
print("  Cores    :", spark.sparkContext.defaultParallelism)
print("  Python   :", conf.get("spark.pyspark.python"))

# Comprueba el worker de Python (createDataFrame lo necesita; range() no).
n_worker = spark.createDataFrame([(1,)], ["ok"]).count()
print("  Worker   : OK (", n_worker, "fila )")


c:\Users\JORGE\miniconda3\envs\py_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


SparkSession lista (modo LOCAL)
  App      : Unidad2-Clase1-PySpark
  Master   : local[*]
  Version  : 4.2.0
  Driver   : 127.0.0.1
  UI       : http://127.0.0.1:4040
  Cores    : 8
  Python   : c:\Users\JORGE\miniconda3\envs\py_env\python.exe
  Worker   : OK ( 1 fila )


In [3]:
#Verficacion de pereza
numeros = spark.range(0, 1_000_000)

pares = numeros.filter(F.col("id") % 2 == 0)

print("Tipo de 'pares':", type(pares).__name__)
print("HAsta aquí SPark no ha reccorrido aun el millon (transformación)")

cuantos = pares.count()
print("PAres entre 0 y 999999", cuantos)

Tipo de 'pares': DataFrame
HAsta aquí SPark no ha reccorrido aun el millon (transformación)
PAres entre 0 y 999999 500000


## DataFrames de PySpark

In [4]:
# Primer dataframe en pyspark

personas = spark.createDataFrame(
    [
        (1, "Julio", 31, "Espana"),
        (2, "Jorge", 28, "Colombia"),
        (3, "Marta", 41, "Espana"),
        (4, "Diego", 35, "Mexico"),
    ],
    schema=["id", "nombre", "edad", "pais"]
)

print("Columnas :", personas.columns)
print("Num filas:", personas.count())
print()
print("Esquema (nombre + tipo Spark):")
personas.printSchema()
print("Primeras filas (accion show)")
personas.show()

Columnas : ['id', 'nombre', 'edad', 'pais']
Num filas: 4

Esquema (nombre + tipo Spark):
root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- pais: string (nullable = true)

Primeras filas (accion show)
+---+------+----+--------+
| id|nombre|edad|    pais|
+---+------+----+--------+
|  1| Julio|  31|  Espana|
|  2| Jorge|  28|Colombia|
|  3| Marta|  41|  Espana|
|  4| Diego|  35|  Mexico|
+---+------+----+--------+



# CSV y PARQUET

In [8]:
# RAIZ ya es .../Modulo_2/Unidad_2 (celda de comprobacion).
# No hay que volver a poner Modulo_2/Unidad_2 o Spark busca un path que no existe.

ruta_ventas_csv = (RAIZ / "data" / "row" / "ventas.csv").resolve()
print("CSV :", ruta_ventas_csv)
print("Existe:", ruta_ventas_csv.exists())

ventas_inferido = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_ventas_csv.as_posix())
)

print("Filas (accion count):", ventas_inferido.count())
print("Particiones:", ventas_inferido.rdd.getNumPartitions())
print()
ventas_inferido.printSchema()
ventas_inferido.show(5, truncate=False)

CSV : C:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_2\data\row\ventas.csv
Existe: True


Filas (accion count): 200000
Particiones: 3

root
 |-- venta_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- producto_id: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- descuento: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)

+--------+----------+-----------+-----------+--------+---------------+---------+-------+-----+
|venta_id|cliente_id|producto_id|fecha_venta|cantidad|precio_unitario|descuento|total  |canal|
+--------+----------+-----------+-----------+--------+---------------+---------+-------+-----+
|1       |2893      |1812       |2025-08-04 |2       |977.23         |0.2      |1563.57|web  |
|2       |1187      |1563       |2025-09-03 |3       |1083.68        |0.0      |3251.04|app  |
|3       |1053      |1184       |2025-11-02 |2       |134.73         |0.0      |269.46 |web  |
|4     